# ASN dense-subgraph reproduction — data generation

The original `gen_reallocation_new_train_test_asn_graph_sampling.ipynb` samples ASN train/test
subgraphs uniformly at random from the 1739-node `ASN2k.json` topology. Its leftover
`train_98_nodes.npy` (98 nodes, 202 edges, avg out-degree 2.06) trains very poorly with this
pipeline: validation objgap plateaus around 15-25% no matter the learning rate, epoch count,
model capacity, or loss weighting (all tried — see README).

Root cause: this is a **random** 98-node sample of a *sparse* 1739-node backbone graph, so it's
much sparser than what the model was actually designed/tuned for -- e.g. B4 (12 nodes, 38 edges,
avg out-degree 3.17) trains cleanly with the same code. A random 98-node ASN sample has too little
path diversity between (s,t) pairs: k=4 shortest paths overlap heavily, so the LP's capacity
constraint rows are highly correlated, which starves the IPM-imitation training signal.

**Fix used here:** keep the node *count* (98) and the exact same LP-generation code
(`generate_reallocation`, `k=4`, `number_of_st=10`), but pick a *denser* 98-node subgraph of the
same ASN2k backbone -- found via a greedy k-core-style peel (repeatedly drop the lowest-degree
node from the full graph until 98 remain), instead of `np.random.choice`.


In [ ]:
import json
import numpy as np
import networkx as nx

root = 'raw/'
G = json.load(open(root + 'asn_graph/ASN2k.json'))
g = nx.DiGraph()
g.add_nodes_from([i['id'] for i in G['nodes']])
g.add_edges_from([(i['source'], i['target']) for i in G['links']])
und = g.to_undirected()
print('full ASN2k graph:', g.number_of_nodes(), 'nodes,', g.number_of_edges(), 'edges')


## 1. Find a dense, strongly-connected 98-node subgraph (greedy k-core peel)


In [ ]:
def greedy_peel_to_size(und, target_size, rng, restarts=8):
    """Repeatedly remove the lowest-degree node until `target_size` nodes remain;
    keep the densest strongly-connected result over a few random tie-break restarts."""
    best = None
    for r in range(restarts):
        h = und.copy()
        while h.number_of_nodes() > target_size:
            deg = dict(h.degree())
            min_deg = min(deg.values())
            candidates = [n for n, d in deg.items() if d == min_deg]
            rm = candidates[rng.randint(len(candidates))]
            h.remove_node(rm)
        sub = g.subgraph(h.nodes())
        if nx.is_strongly_connected(sub):
            ne = sub.number_of_edges()
            if best is None or ne > best[0]:
                best = (ne, list(h.nodes()))
    return best

# NOTE: the shipped raw_asn_saved/*.pkl.gz and checkpoints/best_model.pt were generated
# from a slightly less exhaustive BFS-based search (1024 edges, avg out-degree 10.4);
# this greedy peel typically finds an even denser one (~1300 edges) -- either works, but
# to exactly reproduce the shipped checkpoint use raw/asn_graph/dense98_nodes.npy directly
# (next cell) rather than re-running this search.
rng = np.random.RandomState(2024)
# best = greedy_peel_to_size(und, 98, rng, restarts=10)
# nodes98 = np.array(best[1])

# reproduce the exact shipped topology:
nodes98 = np.load(root + 'asn_graph/dense98_nodes.npy')
sub_check = g.subgraph(nodes98)
print('using dense98_nodes.npy:', sub_check.number_of_nodes(), 'nodes,',
      sub_check.number_of_edges(), 'edges, avg_out_degree=',
      sub_check.number_of_edges() / sub_check.number_of_nodes())
print('strongly connected:', nx.is_strongly_connected(sub_check))


## 2. Generate LP instances on this subgraph

Identical LP construction to the original notebook's `generate_reallocation` (`k=4` shortest paths, `number_of_st=10` demand pairs, demand in `[1000, 5000]`) -- only the node set differs.


In [ ]:
from itertools import islice
from scipy.linalg import LinAlgWarning
from scipy.optimize._optimize import OptimizeWarning
import random, time, gzip, pickle, warnings
import torch
from solver.linprog import linprog

def k_shortest_paths(G, source, target, k, weight=None):
    return list(islice(nx.shortest_simple_paths(G, source, target, weight=weight), k))

def generate_reallocation(G, STD, Pd, k):
    A1 = []
    for i in range(len(STD)):
        a = np.zeros(len(STD) * k)
        a[k*i: k*i+k] = 1
        A1.append(a)
    A1 = np.array(A1)
    b1 = np.ones(len(STD))
    edges_list = list(G.edges())
    A2 = np.zeros((G.number_of_edges(), len(STD) * k))
    for i in range(len(STD)):
        paths = Pd[tuple(STD[i][0])]
        for j in range(k):
            p = paths[j]
            for n in range(len(p) - 1):
                if (p[n], p[n+1]) in edges_list:
                    A2[edges_list.index((p[n], p[n+1]))][k*i+j] = STD[i][1]
    b2 = np.array(list(nx.get_edge_attributes(G, 'weight').values()))
    zero_row_indices = np.where(A2.any(axis=1) == 0)[0]
    A2 = np.delete(A2, zero_row_indices, axis=0)
    b2 = np.delete(b2, zero_row_indices, axis=0)
    for i in range(A2.shape[0]):
        A2[i] = A2[i] / b2[i]
        b2[i] = b2[i] / b2[i]
    c = -1 * np.concatenate([np.ones(k) * STD[i][1] for i in range(len(STD))])
    return A1, b1, A2, b2, c


In [ ]:
def gen_instances(num, seed, sub_nodes_w, sub_nodes_noC, k=4, min_d=1000, max_d=5000,
                   number_of_st=10, bounds=(0., 1.)):
    random.seed(seed); np.random.seed(seed)
    warnings.filterwarnings('error')
    ips, success_cnt, fail_cnt, attempts = [], 0, 0, 0
    max_attempts = num * 20
    t0 = time.time()
    while success_cnt < num and attempts < max_attempts:
        attempts += 1
        std, Pd, count_std, tries, ok = [], {}, 0, 0, True
        while count_std != number_of_st:
            tries += 1
            if tries > 2000:
                ok = False; break
            st = np.random.choice(sub_nodes_w.nodes(), 2, replace=False)
            d = random.uniform(min_d, max_d)
            try:
                k_paths = k_shortest_paths(sub_nodes_noC, st[0], st[1], k=k)
            except Exception:
                continue
            if len(k_paths) != k:
                continue
            Pd[(st[0], st[1])] = k_paths
            std.append((st, d)); count_std += 1
        if not ok:
            fail_cnt += 1; continue
        A1, b1, A2, b2, c = generate_reallocation(sub_nodes_w, std, Pd, k)
        A = np.vstack([A1, A2]); b = np.hstack([b1, b2])
        try:
            res = linprog(c, A_ub=A, b_ub=b, A_eq=None, b_eq=None, bounds=bounds,
                          method='interior-point')
        except (LinAlgWarning, OptimizeWarning, AssertionError):
            fail_cnt += 1; continue
        if res.success and not np.isnan(res.fun):
            ips.append((torch.from_numpy(A).to(torch.float),
                        torch.from_numpy(b).to(torch.float),
                        torch.from_numpy(c).to(torch.float)))
            success_cnt += 1
    warnings.resetwarnings()
    print(f'requested={num} success={success_cnt} fail={fail_cnt} attempts={attempts} '
          f'time={time.time()-t0:.1f}s')
    return ips


In [ ]:
asn_graph_weight = nx.DiGraph()
asn_graph_weight.add_nodes_from([i['id'] for i in G['nodes']])
for i in G['links']:
    asn_graph_weight.add_edge(i['source'], i['target'], weight=i['capacity'])

sub_w = asn_graph_weight.subgraph(nodes98)
sub_noC = g.subgraph(nodes98)

# train: 400 instances, seed=2024
train_ips = gen_instances(400, 2024, sub_w, sub_noC)
with gzip.open('raw_asn_saved/instance_train_dense98.pkl.gz', 'wb') as f:
    pickle.dump(train_ips, f)

# test: 150 instances, seed=2025 (different seed -> different demand/path draws, same topology)
test_ips = gen_instances(150, 2025, sub_w, sub_noC)
with gzip.open('raw_asn_saved/instance_test_dense98.pkl.gz', 'wb') as f:
    pickle.dump(test_ips, f)


## 3. Cache as LPDataset

Same as B4: stage each split into `raw/raw/` (LPDataset's expected input dir), build the
cache, then move the staged file out of the way before the next split -- `raw/raw/` gets
overwritten between splits, `raw_asn_saved/` is the permanent copy.


In [ ]:
import shutil, os
from torch_geometric.transforms import Compose
from data.data_preprocess import HeteroAddLaplacianEigenvectorPE, SubSample
from data.dataset import LPDataset

# PyTorch 2.6 defaults torch.load(weights_only=True), which breaks loading PyG
# Batch/HeteroData objects from the processed cache -- patch globally.
import torch as _t
_orig_load = _t.load
_t.load = lambda *a, **k: _orig_load(*a, **{**k, 'weights_only': False})

ipm = 16
pre = Compose([HeteroAddLaplacianEigenvectorPE(k=0), SubSample(ipm)])

def stage_and_cache(src, extra_path):
    if os.path.isdir('raw/raw'):
        shutil.rmtree('raw/raw')
    os.makedirs('raw/raw')
    shutil.copy(src, 'raw/raw/instance_0.pkl.gz')
    ds = LPDataset('raw', extra_path=f'1restarts_0lap_{ipm}steps_upper_{extra_path}',
                    upper_bound=1, rand_starts=1, pre_transform=pre)
    print(f'{extra_path}: cached {len(ds)} instances')

stage_and_cache('raw_asn_saved/instance_train_dense98.pkl.gz', 'train_dense98')
stage_and_cache('raw_asn_saved/instance_test_dense98.pkl.gz', 'test_dense98')
